# Slope vs. RMSE per trajectory, variable wind

One point per trajectory. The y-axis is the weighted |slope| (rad/s), the same metric as panel C. It uses `uaa.get_weighted_value`, which weights the bootstrap iterations of each trajectory by 1/rmse_affine².

The x-axis depends on `X_METRIC`:
- `'r2_to_mean_model'` (default): each trajectory's circular R² (`uaa.circular_r2`) against the **average model** of its panel, over aligned time −1 to +1 s. R² < 0 means the model does worse than the trajectory's own mean direction; this happens for trajectories that barely turn while the model predicts turning, and the values are unbounded below. To compress that tail without a hard clip, the axis plots `R2_POS_SCALE`·R² for R² ≥ 0 and −ln(1 − R²) for R² < 0. The two pieces are continuous at 0; the positive side is stretched to give the well-fit trajectories more room. Tick labels are in R² units.
- `'rmse_to_mean_model'`: the RMSE between each trajectory's aligned course and the **average model** of its panel. The average model is the circular mean of the aligned affine fits, the magenta line in panels A/B, over aligned time −1 to +1 s.
- `'rmse_affine'`: the trajectory's own weighted affine-fit RMSE.

Writes the `slope_rmse_flash` and `slope_rmse_sham` layers into the shared figure SVG.

Data: flash comes from `varwind_N700_flash_*` (same as panel A); sham comes from `var_wind_kevin_analyzed/*_SHAM` (same as panels B and C).

Overlay: a 95% confidence ellipse for the joint (median x, median slope) of each panel. It is estimated by bootstrapping trajectories, computing both medians for each resample, and fitting a χ²(2) ellipse to the bootstrap distribution in plotted coordinates. The panel's median is marked with a dot.

Period shift: the upwind surge before the circling sits mostly before the fit window, but at higher wind speeds it spills into the window's start. With `PERIOD_SHIFTS = [0, 1]`, each trajectory is scored twice against the same average model:
- at its default alignment;
- shifted left by one full period of its own best fit (2π/|slope|), i.e. using the course one period later.

Because the fit is periodic, this keeps the trajectory's phase relative to its fit. The shift with the higher R² is kept, and the RMSE comes from the same shift. Taking the better of two scores biases R² slightly upward, and this applies equally to flash and sham.

In [ ]:
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Helvetica'] + matplotlib.rcParams['font.sans-serif']
matplotlib.rcParams['font.size'] = 6
matplotlib.rcParams['text.usetex'] = False
matplotlib.rcParams["ps.usedistiller"] = 'xpdf'
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.weight'] = 'normal'
matplotlib.rcParams["mathtext.fontset"] = 'cm'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

import figurefirst as fifi

from align_course_direction_analysis import unifying_algo_analysis as uaa
from align_course_direction_analysis.util import angle_distance, mean_angle
from splitflow.set_zorder_functions import *

In [ ]:
TRANSLATION = True
FIGURE_NAME = 'supplemental_unifying_analysis_variable_wind_unifying.svg'
DATA_DIR = Path('../../../Data/unifying_algo_results')
TRAJEC_FILENAME = Path('../../../Data/Experimental_Fly_Data') / 'flies_varwind_C1WT_combined_optotrigger_fancontrol_trimmed.parquet'

# x-axis metric: 'r2_to_mean_model', 'rmse_to_mean_model' or 'rmse_affine'
X_METRIC = 'r2_to_mean_model'
X_LABELS = {'r2_to_mean_model': 'Circular $R^2$ to mean model',
            'rmse_to_mean_model': 'RMSE to mean model (rad)',
            'rmse_affine': 'RMSE (rad)'}

# Average model window: columns of the aligned arrays (-300..300 samples at 0.01 s),
# 200:400 = aligned time -1..1 s, same as the mean-model line in panels A/B and the MI in panel D
ROI = slice(200, 400)
MIN_SAMPLES = 50  # trajectories with fewer valid samples in the ROI are dropped (0.5 s)

# Temporal shifts (in periods of each trajectory's own best fit, shifted left) to try; the best R2 is kept.
# [0] = default alignment only
PERIOD_SHIFTS = [0, 1]

CONDITIONS = ['flash', 'sham']
SPEEDS = ['12', '20', '23', '28', '40', '90']
SPEED_LABELS = {'12': 0, '20': 3, '23': 10, '28': 20, '40': 30, '90': 60}  # cm/s

# One color per condition (no per-speed highlight)
COLORS = {'flash': '#a50f15',  # dark crimson
          'sham': 'gray'}
def get_color(condition, speed):
    return COLORS[condition]

MARKERSIZE = 0.75
ALPHA = 0.5

# Confidence ellipse around the (median x, median slope)
ELLIPSE_CONFIDENCE = 0.95
N_BOOTSTRAP = 2000
ELLIPSE_ALPHA = 0.3       # fill transparency
RNG_SEED = 0              # for reproducible bootstrap

# Shared axis limits for all panels
R2_POS_SCALE = 3.0  # stretch factor for R2 >= 0 on the plotted axis
XLIMS = {'r2_to_mean_model': (-5.5, R2_POS_SCALE),  # in transformed units; -5.5 corresponds to R2 of about -244
         'rmse_to_mean_model': (0, 3.0),  # angular RMSE is bounded by pi
         'rmse_affine': (0, 1.5)}
XLIM = XLIMS[X_METRIC]

# Tick positions/labels for the R2 axis, given in R2 units
R2_TICKS = [-100, -10, -1, 0, 0.5, 1]
R2_TICKLABELS = ['-100', '-10', '', '0', '', '1']
YLIM = (0, 8)    # weighted |slope| (rad/s), same as panel C

In [ ]:
# Sham uses Kevin's analyzed files (same as panels B and C); flash uses the N700 files (same as panel A)
SHAM_FILENAMES = {'12': 'var-wind-12-fan_0-cms_all-traj_align_SHAM.parquet',
                  '20': 'var-wind-20-fan_0-03-cms_all-traj_align_SHAM.parquet',
                  '23': 'var-wind-23-fan_0-1-cms_all-traj_align_SHAM.parquet',
                  '28': 'var-wind-28-fan_0-2-cms_all-traj_align_SHAM.parquet',
                  '40': 'var-wind-40-fan_0-3-cms_all-traj_align_SHAM.parquet',
                  '90': 'var-wind-90-fan_0-6-cms_all-traj_align_SHAM.parquet'}

def get_unifying_filename(condition, speed):
    if condition == 'sham':
        return DATA_DIR / 'var_wind_kevin_analyzed' / SHAM_FILENAMES[speed]
    return DATA_DIR / ('varwind_N700_' + condition + '_' + speed + '.0_translation' + str(TRANSLATION) + '.parquet')

In [ ]:
def get_weighted_slopes(unifying_algo_data):
    """Weighted |slope| (rad/s) and weighted rmse_affine per trajectory, keyed by obj_id."""
    timestep = unifying_algo_data.timestep_sec.iloc[0]  # 0.01 s; converts rad/sample -> rad/s
    slopes = {}
    rmse_affine = {}
    for obj_id in unifying_algo_data.obj_id_unique_event.unique():
        data = unifying_algo_data[unifying_algo_data.obj_id_unique_event==obj_id]
        # get_weighted_value takes abs() of col, weights = 1/rmse_affine**2
        slopes[obj_id] = uaa.get_weighted_value(data, col='slope', use='rmse_affine') / timestep
        rmse_affine[obj_id] = uaa.get_weighted_value(data, col='rmse_affine', use='rmse_affine')
    return slopes, rmse_affine


def r2_axis_transform(r2):
    """
    R2_POS_SCALE*R2 for R2 >= 0, -ln(1 - R2) for R2 < 0. Continuous at 0;
    compresses the unbounded negative tail smoothly instead of clipping,
    and stretches the positive side to give well-fit trajectories more room.
    """
    r2 = np.asarray(r2, dtype=float)
    return np.where(r2 >= 0, R2_POS_SCALE * r2, -np.log(1 - np.minimum(r2, 0)))


def get_fit_to_mean_model(unifying_algo_data, trajec_df):
    """
    RMSE and circular R2 between each aligned trajectory and the average model (mean of the aligned affine fits)
    over ROI, following the mean-model construction in AB_varwind_figure_align_course_directions
    and the per-trajectory comparison in sup_fig_unifying_analysis/C_MI_plots_flies_allwtsplitflow.
    """
    obj_ids = unifying_algo_data.obj_id_unique_event.unique()
    trajec_df = trajec_df[trajec_df.obj_id_unique_event.isin(obj_ids)]  # speeds up per-trajectory lookups

    results = uaa.get_aligned_course_array(unifying_algo_data, trajec_df, return_linear_and_affine_fits=True)
    ix_master, aligned_time_since_flash_arr, aligned_course_arr, aligned_roi_arr, aligned_linear_fits, aligned_affine_fits = results

    mean_affine_fit = np.array([mean_angle(aligned_affine_fits[:,i]) for i in range(aligned_affine_fits.shape[1])])
    mean_roi = mean_affine_fit[ROI]
    n_cols = aligned_course_arr.shape[1]

    rmse = {}
    r2 = {}
    shift_used = {}
    for i, obj_id in enumerate(obj_ids):  # rows follow obj_ids order
        # period (in samples) of this trajectory's best fit, the same fit used for its alignment
        fits = unifying_algo_data[unifying_algo_data.obj_id_unique_event==obj_id].sort_values('rmse_affine')
        period = int(np.round(2 * np.pi / np.abs(fits.slope.iloc[0])))

        best = (np.nan, np.nan, np.nan)  # (r2, rmse, n_periods)
        for n_periods in PERIOD_SHIFTS:
            # shifting the trajectory left by n periods = comparing its course n periods later to the model
            start = ROI.start + n_periods * period
            stop = ROI.stop + n_periods * period
            course_win = np.full(stop - start, np.nan)
            src = np.arange(start, stop)
            ok_src = src < n_cols
            course_win[ok_src] = aligned_course_arr[i, src[ok_src]]

            ix = ~np.isnan(course_win) & ~np.isnan(mean_roi)
            if ix.sum() < MIN_SAMPLES:
                continue
            r2_val = uaa.circular_r2(course_win[ix], mean_roi[ix])
            if np.isnan(best[0]) or r2_val > best[0]:
                rmse_val = np.sqrt(np.mean(angle_distance(course_win[ix], mean_roi[ix])**2))
                best = (r2_val, rmse_val, n_periods)

        r2[obj_id], rmse[obj_id], shift_used[obj_id] = best
    return rmse, r2, shift_used

In [ ]:
trajec_df = pd.read_parquet(TRAJEC_FILENAME)

results = {}
for condition in CONDITIONS:
    for speed in SPEEDS:
        unifying_algo_data = pd.read_parquet(get_unifying_filename(condition, speed))
        slopes, rmse_affine = get_weighted_slopes(unifying_algo_data)
        if X_METRIC == 'rmse_affine':
            x_values = rmse_affine
        else:
            rmse_to_mean, r2_to_mean, shift_used = get_fit_to_mean_model(unifying_algo_data, trajec_df)
            x_values = r2_to_mean if X_METRIC == 'r2_to_mean_model' else rmse_to_mean

        obj_ids = list(slopes.keys())
        x = np.array([x_values[o] for o in obj_ids])
        y = np.array([slopes[o] for o in obj_ids])
        x_affine = np.array([rmse_affine[o] for o in obj_ids])
        ok = ~np.isnan(x)
        x_plot = r2_axis_transform(x) if X_METRIC == 'r2_to_mean_model' else x
        results[(condition, speed)] = {'slopes': y[ok], 'rmse': x_plot[ok], 'x_raw': x[ok]}

        r = results[(condition, speed)]
        n_out = np.sum((r['rmse']<XLIM[0]) | (r['rmse']>XLIM[1]) | (r['slopes']>YLIM[1]))
        print(condition, speed, 'n =', ok.sum(), '(dropped %d)' % (~ok).sum(),
              ' median slope = %.2f' % np.median(r['slopes']),
              ' median x = %.2f' % np.median(r['x_raw']),
              ' (median rmse_affine = %.2f)' % np.median(x_affine[ok]),
              ' x range = %.2f to %.2f' % (r['x_raw'].min(), r['x_raw'].max()),
              ' outside limits:', n_out)
        if X_METRIC != 'rmse_affine':
            s = np.array([shift_used[o] for o in obj_ids])[ok]
            print('      fraction using each period shift:',
                  {p: round(float(np.mean(s == p)), 2) for p in PERIOD_SHIFTS})

In [ ]:
import scipy.stats
from matplotlib.patches import Ellipse

rng = np.random.default_rng(RNG_SEED)

def plot_median_confidence_ellipse(ax, x, y, color, confidence=ELLIPSE_CONFIDENCE, n_bootstrap=N_BOOTSTRAP, alpha=ELLIPSE_ALPHA):
    """
    Confidence ellipse for the joint (median x, median y): bootstrap trajectories, take both medians
    per resample, then draw the chi2(2) ellipse of the bootstrap covariance, centered on the observed medians.
    x should already be in plotted coordinates.
    """
    x = np.asarray(x); y = np.asarray(y)
    n = len(x)
    ix = rng.integers(0, n, size=(n_bootstrap, n))
    boot = np.column_stack([np.median(x[ix], axis=1), np.median(y[ix], axis=1)])

    center = np.array([np.median(x), np.median(y)])
    cov = np.cov(boot, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    eigvals = np.clip(eigvals, 0, None)
    scale = np.sqrt(scipy.stats.chi2.ppf(confidence, df=2))
    width, height = 2 * scale * np.sqrt(eigvals[::-1])
    angle = np.degrees(np.arctan2(eigvecs[1, -1], eigvecs[0, -1]))

    ax.add_patch(Ellipse(center, width, height, angle=angle,
                         facecolor=color, alpha=alpha, edgecolor='none', zorder=10))
    ax.add_patch(Ellipse(center, width, height, angle=angle,
                         facecolor='none', edgecolor=color, linewidth=0.5, zorder=11))
    ax.plot(*center, 'o', markersize=1.5, color=color, markeredgecolor='none', zorder=12)

    # dashed guide lines from the median to the x and y axes
    ax.plot([XLIM[0], center[0]], [center[1], center[1]], linestyle=(0, (2, 1.5)),
            color=color, linewidth=0.5, zorder=9)
    ax.plot([center[0], center[0]], [YLIM[0], center[1]], linestyle=(0, (2, 1.5)),
            color=color, linewidth=0.5, zorder=9)
    return center, width, height


def clean_labels(ax, show_labels):
    ax.set_xlim(*XLIM)
    ax.set_ylim(*YLIM)

    if X_METRIC == 'r2_to_mean_model':
        ax.set_xticks(r2_axis_transform(R2_TICKS))
    else:
        ax.set_xticks(np.arange(XLIM[0], XLIM[1] + 1e-9, 0.5))
    ax.set_yticks([0, 2, 4, 6, 8])

    if show_labels:
        if X_METRIC == 'r2_to_mean_model':
            ax.set_xticklabels(R2_TICKLABELS)
        else:
            xticks = ax.get_xticks()
            ax.set_xticklabels(['%g' % xticks[0]] + [''] * (len(xticks) - 2) + ['%g' % xticks[-1]])
        ax.set_yticklabels(['0', '', '', '', '8'])
        ax.set_xlabel(X_LABELS[X_METRIC], labelpad=1)
        ax.set_ylabel('Slope (rad/s)', labelpad=-2)
    else:
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.set_xlabel('')
        ax.set_ylabel('')

    ax.tick_params(axis='y', pad=2)
    ax.tick_params(axis='x', pad=2)

    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                     tick_length=2.5,
                                     spine_locations={'left': 5, 'bottom': 5},
                                     linewidth=0.5)
    fifi.mpl_functions.set_fontsize(ax, 6)

In [ ]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

In [ ]:
for condition in CONDITIONS:
    for speed in SPEEDS:
        ax = layout.axes[('slope_rmse_' + condition, speed)]
        r = results[(condition, speed)]


        ax.plot(r['rmse'], r['slopes'], 'o',
                markersize=MARKERSIZE,
                color=get_color(condition, speed),
                markeredgecolor='none',
                alpha=ALPHA)

        plot_median_confidence_ellipse(ax, r['rmse'], r['slopes'], color=get_color(condition, speed))

        clean_labels(ax, show_labels=(speed == SPEEDS[0]))
        set_selective_rasterization(ax, rasterize_markers=['o'])

In [ ]:
for condition in CONDITIONS:
    fifi_figure_label = 'slope_rmse_' + condition
    layout.append_figure_to_layer(layout.figures[fifi_figure_label], fifi_figure_label, cleartarget=True)
layout.write_svg(FIGURE_NAME)